In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
excel_solar = snakemake.input.excel_solar
excel_wind = snakemake.input.excel_wind
excel_nuclear = snakemake.input.excel_nuclear

gem_output_path = snakemake.output.existing_cap

euroshape = gpd.read_file(snakemake.input.onshore_shapefile).set_index(["index"])
zones = euroshape.CNTR_CODE.unique().tolist()  # get relevant countries


In [ ]:
# Technologies used in GEM
pv_types = ["PV", "Assumed PV"]
Offshore_types = ["Offshore mount unknown", "Offshore hard mount", "Offshore floating"]

In [ ]:
country_codes = {
    "Albania": "AL",
    "Austria": "AT",
    "Bosnia and Herzegovina": "BA",
    "Belgium": "BE",
    "Bulgaria": "BG",
    "Switzerland": "CH",
    "Cyprus": "CY",
    "Czech Republic": "CZ",
    "Germany": "DE",
    "Denmark": "DK",
    "Estonia": "EE",
    "Spain": "ES",
    "Finland": "FI",
    "France": "FR",
    "United Kingdom": "UK",
    "Greece": "GR",
    "Croatia": "HR",
    "Hungary": "HU",
    "Ireland": "IE",
    "Iceland": "IS",
    "Italy": "IT",
    "Liechtenstein": "LI",
    "Lithuania": "LT",
    "Luxembourg": "LU",
    "Latvia": "LV",
    "Montenegro": "ME",
    "North Macedonia": "MK",
    "Malta": "MT",
    "Netherlands": "NL",
    "Norway": "NO",
    "Poland": "PL",
    "Portugal": "PT",
    "Romania": "RO",
    "Serbia": "RS",
    "Sweden": "SE",
    "Slovenia": "SI",
    "Slovakia": "SK",
    "Kosovo": "XK",
}

In [ ]:
### Solar
gem_solar = pd.concat(
    pd.read_excel(
        excel_solar,
        sheet_name=["20 MW+", "1-20 MW"],
        skiprows=0,
    )
)
gem_solar = (
    gem_solar.rename(
        columns={
            "Project Name": "project_name",
            "Country/Area": "country",
            "Technology Type": "Technology",
            "Capacity (MW)": "capacity_MW",
            "Status": "status",
            "Latitude": "lat",
            "Longitude": "lon",
            "Start year": "commissioning_year",
            "Retired year": "retirement_year",
        }
    )
    .query("Technology == @pv_types")
    .query("lat.notna() & lon.notna()")
    .drop_duplicates(subset=["project_name", "capacity_MW"])
    .loc[
        :,
        [
            "project_name",
            "country",
            "capacity_MW",
            "status",
            "lat",
            "lon",
            "commissioning_year",
            "retirement_year",
        ],
    ]
)
### Wind

gem_wind = pd.concat(
    pd.read_excel(
        excel_wind,
        sheet_name=["Data", "Below Threshold"],
        skiprows=0,
    )
)
gem_wind = (
    gem_wind.rename(
        columns={
            "Project Name": "project_name",
            "Country/Area": "country",
            "Installation Type": "Technology",
            "Capacity (MW)": "capacity_MW",
            "Status": "status",
            "Latitude": "lat",
            "Longitude": "lon",
            "Start year": "commissioning_year",
            "Retired year": "retirement_year",
        }
    )
    .query("lat.notna() & lon.notna()")
    .drop_duplicates(subset=["project_name", "capacity_MW"])
    .loc[
        :,
        [
            "project_name",
            "country",
            "capacity_MW",
            "Technology",
            "status",
            "lat",
            "lon",
            "commissioning_year",
            "retirement_year",
        ],
    ]
)

gem_onshore = (
    gem_wind.query("Technology == 'Onshore'")
    .drop("Technology", axis=1)
    .drop_duplicates(subset=["project_name", "capacity_MW"])
)

gem_offshore = (
    gem_wind.query("Technology == @Offshore_types")
    .drop("Technology", axis=1)
    .drop_duplicates(subset=["project_name", "capacity_MW"])
)
### Nuclear
gem_nuclear = pd.concat(
    pd.read_excel(
        excel_nuclear,
        sheet_name=["Data"],
        skiprows=0,
    )
)

gem_nuclear = gem_nuclear.rename(
    columns={
        "Project Name": "project_name",
        "Start Year": "commissioning_year",
        "Construction Start Date": "construction_start",
        "Retirement Year": "retirement_year",
        "Cancellation Year": "cancellation_year",
        "Capacity (MW)": "capacity_MW",
        "Country/Area": "country",
        "Status": "status",
        "Latitude": "lat",
        "Longitude": "lon",
    }
).loc[
    :,
    [
        "country",
        "capacity_MW",
        "project_name",
        "status",
        "lat",
        "lon",
        "commissioning_year",
        "construction_start",
        "retirement_year",
        "cancellation_year",
    ],
]


In [ ]:
# Merge onshore, offshore, solar, and nuclear
merged_gem = pd.concat(
    [
        (gem_onshore.assign(Technology="onshore")),
        (gem_offshore.assign(Technology="offshore")),
        (gem_solar.assign(Technology="solar")),
        (
            gem_nuclear
            # These columns are not relevant
            .drop(columns=["construction_start", "cancellation_year"]).assign(
                Technology="nuclear"
            )
        ),
    ]
)

merged_gem = (
    merged_gem
    # Add country code
    .assign(iso_code=merged_gem["country"].replace(country_codes))
    # Only select the relevant countries
    .query("iso_code == @zones")
)


In [ ]:
# Calculate lifetime of Solar, Onshore, Offshore and Nuclear based on GEM data
# with retirement year and commissiong year

lifetime = (
    merged_gem.query("commissioning_year.notnull() & retirement_year.notnull()")
    .assign(lifetime_year=lambda x: x.retirement_year - x.commissioning_year)
    .loc[:, ["country", "Technology", "lifetime_year"]]
)

# lifetime.groupby(["country","Technology"]).describe()
lifetime.groupby(["Technology"]).describe()


In [ ]:
# Only plants with status "operating"
filtered_gem = (
    merged_gem.query("status == ['operating','construction']")
    .loc[
        :,
        [
            "project_name",
            "Technology",
            "status",
            "country",
            "iso_code",
            "lat",
            "lon",
            "capacity_MW",
            "commissioning_year",
        ],
    ]
    .reset_index(drop=True)
)


In [ ]:
# Estimate decomissionned year based on historical data (lifeftime) by technology

lifetime_tech_dict = {
    "nuclear": (
        lifetime.query("Technology=='nuclear'").loc[:, "lifetime_year"].quantile(0.75)
    ),
    "onshore": 30,
    "offshore": 30,
    "solar": 30,
}

# Assign lifetime to every technology
gem_dataset = filtered_gem.copy()
for tech, lifetime_value in lifetime_tech_dict.items():
    gem_dataset.loc[gem_dataset["Technology"] == tech, "estimated_retirement_year"] = (
        gem_dataset.loc[gem_dataset["Technology"] == tech, "commissioning_year"]
        + lifetime_value
    )
# Fix format issue and nan values (no commissioning year available)
gem_dataset["commissioning_year"] = (
    gem_dataset["commissioning_year"].fillna(0).round(0).astype(int)
)
gem_dataset["estimated_retirement_year"] = (
    gem_dataset["estimated_retirement_year"].fillna(0).round(0).astype(int)
)


gem_dataset

In [ ]:
# ["project_name","Technology","country","iso_code","lat","lon","capacity_MW","commissioning_year", "estimated_retirement_year"]

# Save dataset
gem_dataset.to_csv(gem_output_path, index=False)